# Buoy raw data processing — bulk parameters

Read every raw buoy file under `inputs/Observations/raw/`, standardize to a
common HyGloW/CF bulk-parameter schema, and write **one NetCDF per buoy**
(`instrument_id`) to `inputs/Observations/processed/` — the layout notebook
10 (`10_Validate_Buoy_Observations.ipynb`) expects.

Scope: **bulk parameters only** (`hs`, `tp`, `dir`, `spr`, plus secondary
fields where the raw data has them). Spectral data that exists for a few
stations is inventoried at the end but not processed here — see
[Spectral data found](#Spectral-data-found-(not-processed-here)).

Matching and standardization logic lives in
[`utils/observation/raw_buoys.py`](utils/observation/raw_buoys.py) — see its
docstring for the two raw formats and the local-time-to-UTC conversion this
notebook applies to the nearshore buoys.

In [1]:
from utils.config import resolve
from utils.observation import onshore_sites as obs
from utils.observation import raw_buoys as rb

## 1 · Match raw files to the observations catalogue

Same matching this case's `00_Buoys_Inspection.ipynb` does for its inventory
— reused here from `utils/observation/raw_buoys.py` so both notebooks stay
in sync.

In [2]:
meta = obs.get_coordinates()
file_index = rb.match_raw_buoy_files(meta)

print(f"matched {len(file_index)}/{len(meta)} catalogue instruments to a raw file")
missing = sorted(set(meta) - set(file_index))
if missing:
    print("catalogue instruments with no raw file found:", missing)

matched 35/35 catalogue instruments to a raw file


## 2 · Standardize and write

For each matched buoy: read the raw file (offshore NetCDF or nearshore
CSV), rename variables to the HyGloW/CF schema, sort/dedupe on time, attach
provenance attributes (source file, institution, network, catalogue
lon/lat/depth), and write `inputs/Observations/processed/{instrument_id}.nc`.

No gap-filling, interpolation, or QC-based filtering is applied — NaNs are
passed through as provided, and the nearshore buoys' own `qflag`/`qcode`/
`percent`/`dof` fields are carried into the output for anyone who wants to
filter downstream.

In [3]:
processed_dir = resolve("inputs/Observations/processed")

rows = []
for iid, info in file_index.items():
    out_path = rb.standardize_and_write(iid, info, meta[iid], processed_dir)
    ds = rb.read_bulk(info)
    rows.append(
        {
            "instrument_id": iid,
            "network": info["network"],
            "n_records": ds.sizes["time"],
            "variables": ", ".join(sorted(ds.data_vars)),
            "has_spr": "spr" in ds.data_vars,
            "hs_nan_pct": round(100 * float(ds["hs"].isnull().mean()), 1),
            "file": out_path.name,
        }
    )

print(f"wrote {len(rows)} buoy file(s) to {processed_dir}")

wrote 35 buoy file(s) to /home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/inputs/Observations/processed


## 3 · Summary

In [4]:
import pandas as pd

summary = pd.DataFrame(rows).set_index("instrument_id").sort_values("network")
summary

,network,n_records,variables,has_spr,hs_nan_pct,file
instrument_id,,,,,,
Narrabeen_Buoy,nearshore (Spotter),3646,"dir, dof, dp, dp_spr, hm0, hmax, hs, percent, ...",True,0.2,Narrabeen_Buoy.nc
Collaroy_Buoy,nearshore (Spotter),3647,"dir, dof, dp, dp_spr, hm0, hmax, hs, percent, ...",True,0.1,Collaroy_Buoy.nc
Collaroy2_Buoy,nearshore (Spotter),1457,"dir, dof, dp, dp_spr, hm0, hmax, hs, percent, ...",True,2.4,Collaroy2_Buoy.nc
Bronte_Buoy,nearshore (Spotter),3238,"dir, dof, dp, dp_spr, hm0, hmax, hs, percent, ...",True,9.2,Bronte_Buoy.nc
Maroubra_Buoy,nearshore (Spotter),3662,"dir, dof, dp, dp_spr, hm0, hmax, hs, percent, ...",True,2.6,Maroubra_Buoy.nc
Figure8Pools_Buoy,nearshore (Spotter),17777,"dir, dof, dp, dp_spr, hm0, hmax, hs, percent, ...",True,0.2,Figure8Pools_Buoy.nc
Woonona_Buoy,nearshore (Spotter),7050,"dir, dof, dp, dp_spr, hm0, hmax, hs, percent, ...",True,0.1,Woonona_Buoy.nc
FairyMeadow_Buoy,nearshore (Spotter),7047,"dir, dof, dp, dp_spr, hm0, hmax, hs, percent, ...",True,22.4,FairyMeadow_Buoy.nc
Gerroa_Buoy,nearshore (Spotter),1853,"dir, dof, dp, dp_spr, hm0, hmax, hs, percent, ...",True,0.0,Gerroa_Buoy.nc


In [5]:
n_offshore = (summary["network"] == "offshore (MHL)").sum()
n_nearshore = (summary["network"] == "nearshore (Spotter)").sum()
n_with_spr = int(summary["has_spr"].sum())

print(f"Buoys processed        : {len(summary)} ({n_offshore} offshore, {n_nearshore} nearshore)")
print(f"Buoys with spr (dir spread): {n_with_spr}/{len(summary)}")
print(f"Offshore buoys lack spr — MHL bulk files never report directional spread.")
print()
print("Variables written per network:")
for network, group in summary.groupby("network"):
    all_vars = sorted(set(v for row in group["variables"] for v in row.split(", ")))
    print(f"  {network}: {all_vars}")

Buoys processed        : 35 (7 offshore, 28 nearshore)
Buoys with spr (dir spread): 28/35
Offshore buoys lack spr — MHL bulk files never report directional spread.

Variables written per network:
  nearshore (Spotter): ['dir', 'dof', 'dp', 'dp_spr', 'hm0', 'hmax', 'hs', 'percent', 'qcode', 'qflag', 'spr', 'tm01', 'tm02', 'tp', 'tz']
  offshore (MHL): ['dir', 'hmax', 'hs', 'tm02', 'tp']


## 4 · Verify

Re-open a couple of the written files directly (not through `raw_buoys`) as
a final sanity check on what actually landed on disk.

In [6]:
import xarray as xr

for iid in ["Sydney_Buoy", "Narrabeen_Buoy"]:
    path = processed_dir / f"{iid}.nc"
    ds = xr.open_dataset(path)
    print(f"=== {iid} ===")
    print(f"  file   : {path.name}")
    print(f"  period : {ds.time.values[0]} to {ds.time.values[-1]}  (UTC)")
    print(f"  vars   : {sorted(ds.data_vars)}")
    print(f"  attrs  : institution={ds.attrs['institution']!r}, network={ds.attrs['network']!r}")
    ds.close()

=== Sydney_Buoy ===
  file   : Sydney_Buoy.nc
  period : 1992-03-02T23:00:00.000003584 to 2017-12-31T12:59:59.999996416  (UTC)
  vars   : ['dir', 'hmax', 'hs', 'tm02', 'tp']
  attrs  : institution='MHL', network='offshore (MHL)'
=== Narrabeen_Buoy ===
  file   : Narrabeen_Buoy.nc
  period : 2016-03-01T23:00:00.000000000 to 2016-05-16T21:30:00.000000000  (UTC)
  vars   : ['dir', 'dof', 'dp', 'dp_spr', 'hm0', 'hmax', 'hs', 'percent', 'qcode', 'qflag', 'spr', 'tm01', 'tm02', 'tp', 'tz']
  attrs  : institution='NSW DCCEEW', network='nearshore (Spotter)'


## Spectral data found (not processed here)

Three offshore stations have raw **spectral** data alongside their bulk
files — out of scope for this notebook (bulk parameters only), flagged here
for a future pass:

| Station | Location | Format |
| --- | --- | --- |
| Sydney | `raw/buoys/WaveSpectra/MHL_Spectra/Sydney/` | 14 yearly `.TXT` files (Datawell directional spectra, 64 freq/dir bins/hour) + 1 `SYDDOW*.DAT` Datawell spectral file under `FromBenModra/` |
| Byron Bay | `raw/buoys/WaveSpectra/MHL_Spectra/Byron Bay/` | 14 yearly `.TXT` files |
| Batemans Bay | `raw/buoys/WaveSpectra/MHL_Spectra/Batemans Bay/` | 12 yearly `.TXT` files |

These are a different, custom MHL text format (not NetCDF/CSV) — parsing
them into `(time, freq, dir)` or `(time, freq)` spectra is a separate task.